#RNN pour IMDB

In [1]:
import os
import pandas as pd

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
# --- CRITICAL: BASE PATH DEFINITION ---
# This is the path to the 'Data' folder containing 'train' and 'test'.
BASE_DIR = '/content/drive/MyDrive/Colab Notebooks/Data'

# --- YOUR REQUESTED PATH VARIABLES ---

# Training Paths
train_pos_path = os.path.join(BASE_DIR, 'train', 'pos')
train_neg_path = os.path.join(BASE_DIR, 'train', 'neg')

# Testing Paths
test_pos_path = os.path.join(BASE_DIR, 'test', 'pos')
test_neg_path = os.path.join(BASE_DIR, 'test', 'neg')


# --- VERIFICATION ---
print("✅ Path variables defined successfully:")
print(f"train_pos_path: {train_pos_path}")
print(f"train_neg_path: {train_neg_path}")
print(f"test_pos_path: {test_pos_path}")
print(f"test_neg_path: {test_neg_path}")

# Run a quick check to ensure the directories actually exist
if os.path.isdir(train_pos_path):
    print("\nDirectory check successful. Ready to load data.")
else:
    print("\n❌ ERROR: Directory not found. Please confirm the BASE_DIR is correct.")

def load_reviews_from_folder(folder, sentiment):
    reviews = []
    for filename in os.listdir(folder):
        if filename.endswith(".txt"):
            with open(os.path.join(folder, filename), "r", encoding="utf-8") as f:
                reviews.append(f.read())
    return pd.DataFrame({"review": reviews, "sentiment": sentiment})

✅ Path variables defined successfully:
train_pos_path: /content/drive/MyDrive/Colab Notebooks/Data/train/pos
train_neg_path: /content/drive/MyDrive/Colab Notebooks/Data/train/neg
test_pos_path: /content/drive/MyDrive/Colab Notebooks/Data/test/pos
test_neg_path: /content/drive/MyDrive/Colab Notebooks/Data/test/neg

Directory check successful. Ready to load data.


In [4]:
df_train_pos = load_reviews_from_folder(train_pos_path, "positive")
df_train_neg = load_reviews_from_folder(train_neg_path, "negative")

df_train = pd.concat([df_train_pos, df_train_neg]).reset_index(drop=True)

In [5]:
df_test_pos = load_reviews_from_folder(test_pos_path, "positive")
df_test_neg = load_reviews_from_folder(test_neg_path, "negative")

df_test = pd.concat([df_test_pos, df_test_neg]).reset_index(drop=True)

In [ ]:
print("Taille train :", len(df_train))
print("Taille test :", len(df_test))

df_train.head()

In [22]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, SimpleRNN, Dense, Dropout
from tensorflow.keras.optimizers import Adam

In [23]:
def clean_text_rnn(text):
    text = text.lower()
    text = re.sub(r"[^a-zA-Z']", " ", text)
    return text

df_train["rnn_review"] = df_train["review"].apply(clean_text_rnn)
df_test["rnn_review"] = df_test["review"].apply(clean_text_rnn)

In [24]:
MAX_WORDS = 20000

tokenizer = Tokenizer(num_words=MAX_WORDS, oov_token="<OOV>")
tokenizer.fit_on_texts(df_train["rnn_review"])

X_train_seq = tokenizer.texts_to_sequences(df_train["rnn_review"])
X_test_seq = tokenizer.texts_to_sequences(df_test["rnn_review"])

In [25]:
MAX_LEN = 200

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN, padding="post", truncating="post")
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN, padding="post", truncating="post")

In [26]:
y_train = (df_train["sentiment"] == "positive").astype(int)
y_test = (df_test["sentiment"] == "positive").astype(int)

In [27]:
model_rnn = Sequential([
    Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LEN),
    SimpleRNN(128, return_sequences=False),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

/usr/local/lib/python3.12/dist-packages/keras/src/layers/core/embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


In [28]:
model_rnn.compile(
    loss="binary_crossentropy",
    optimizer=Adam(learning_rate=0.001),
    metrics=["accuracy"]
)

In [29]:
history = model_rnn.fit(
    X_train_pad,
    y_train,
    epochs=10,
    batch_size=64,
    validation_split=0.2
)

Epoch 1/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 12s 27ms/step - accuracy: 0.5645 - loss: 0.6965 - val_accuracy: 0.0000e+00 - val_loss: 0.9204
Epoch 2/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.5710 - loss: 0.6979 - val_accuracy: 4.0000e-04 - val_loss: 0.9214
Epoch 3/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.5959 - loss: 0.6786 - val_accuracy: 0.0000e+00 - val_loss: 0.9650
Epoch 4/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.6070 - loss: 0.6701 - val_accuracy: 0.0000e+00 - val_loss: 0.9535
Epoch 5/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.6212 - loss: 0.6649 - val_accuracy: 0.0000e+00 - val_loss: 1.0326
Epoch 6/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 20ms/step - accuracy: 0.6272 - loss: 0.6534 - val_accuracy: 0.0112 - val_loss: 1.0549
Epoch 7/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.6488 - loss: 0.6323 - val_accuracy: 0.0718 - val_loss: 0.9713
Epoch 8/10
313/313 ━━━━━━━━━━━━━━━━━━━━ 6s 19ms/step - accuracy: 0.6705 - lo

In [30]:
loss, accuracy = model_rnn.evaluate(X_test_pad, y_test)
print("Accuracy RNN (SimpleRNN) :", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 4s 5ms/step - accuracy: 0.7598 - loss: 0.5990
Accuracy RNN (SimpleRNN) : 0.5192400217056274


#BiLSTM + GloVe

In [31]:
import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, Bidirectional, LSTM, Dense, Dropout
from sklearn.preprocessing import LabelEncoder

In [32]:
label_encoder = LabelEncoder()
y_train_enc = label_encoder.fit_transform(df_train["sentiment"])
y_test_enc = label_encoder.transform(df_test["sentiment"])

In [33]:
MAX_WORDS = 20000
MAX_LEN = 200

tokenizer = Tokenizer(num_words=MAX_WORDS)
tokenizer.fit_on_texts(df_train["cleaned_review"])

X_train_seq = tokenizer.texts_to_sequences(df_train["cleaned_review"])
X_test_seq = tokenizer.texts_to_sequences(df_test["cleaned_review"])

X_train_pad = pad_sequences(X_train_seq, maxlen=MAX_LEN)
X_test_pad = pad_sequences(X_test_seq, maxlen=MAX_LEN)

In [34]:
!wget http://nlp.stanford.edu/data/glove.6B.zip
!unzip glove.6B.zip

--2025-12-18 20:59:37--  http://nlp.stanford.edu/data/glove.6B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.6B.zip [following]
--2025-12-18 20:59:37--  https://nlp.stanford.edu/data/glove.6B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip [following]
--2025-12-18 20:59:37--  https://downloads.cs.stanford.edu/nlp/data/glove.6B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 862182613 (822M) [application/zip]
Saving to: ‘glove.6B.zip’

glov

In [35]:
EMBEDDING_DIM = 100

embeddings_index = {}
with open("glove.6B.100d.txt", encoding="utf-8") as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype="float32")
        embeddings_index[word] = vector

In [36]:
word_index = tokenizer.word_index
embedding_matrix = np.zeros((MAX_WORDS, EMBEDDING_DIM))

for word, i in word_index.items():
    if i < MAX_WORDS:
        vector = embeddings_index.get(word)
        if vector is not None:
            embedding_matrix[i] = vector

In [42]:
model = Sequential([
    Embedding(
        input_dim=MAX_WORDS,
        output_dim=EMBEDDING_DIM,
        weights=[embedding_matrix],
        input_length=MAX_LEN,
        trainable=True
    ),
    Bidirectional(LSTM(128)),
    Dropout(0.5),
    Dense(1, activation="sigmoid")
])

model.compile(
    loss="binary_crossentropy",
    optimizer="adam",
    metrics=["accuracy"]
)

model.summary()

Model: "sequential_5"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_5 (Embedding)         │ ?                      │     2,000,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ bidirectional_2 (Bidirectional) │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_4 (Dropout)             │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_5 (Dense)                 │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 2,000,000 (7.63 MB)

 Trainable params: 2,000,000 (7.63 MB)

 Non-trainable params: 0 (0.00 B)

In [43]:
history = model.fit(
    X_train_pad,
    y_train_enc,
    validation_split=0.2,
    epochs=5,
    batch_size=64
)

Epoch 1/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 11s 27ms/step - accuracy: 0.7482 - loss: 0.5143 - val_accuracy: 0.7672 - val_loss: 0.5265
Epoch 2/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 24ms/step - accuracy: 0.8901 - loss: 0.2831 - val_accuracy: 0.8838 - val_loss: 0.2795
Epoch 3/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 26ms/step - accuracy: 0.9377 - loss: 0.1721 - val_accuracy: 0.8396 - val_loss: 0.4255
Epoch 4/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.9646 - loss: 0.1061 - val_accuracy: 0.7816 - val_loss: 0.6609
Epoch 5/5
313/313 ━━━━━━━━━━━━━━━━━━━━ 8s 25ms/step - accuracy: 0.9824 - loss: 0.0616 - val_accuracy: 0.7810 - val_loss: 0.8695


In [44]:
loss, accuracy = model.evaluate(X_test_pad, y_test_enc)
print("Accuracy BiLSTM + GloVe :", accuracy)

782/782 ━━━━━━━━━━━━━━━━━━━━ 6s 8ms/step - accuracy: 0.8569 - loss: 0.4811
Accuracy BiLSTM + GloVe : 0.8507999777793884
